# Supplementary corpora — cleaning pipeline walkthrough

This notebook walks step-by-step through `02_clean_supplementary.py`, which
cleans the four supplementary corpora (news headlines, Twitter sarcasm,
TweetEval irony, FigLang) into one shared schema so they can be compared
with each other and with the primary SARC corpus.

All the actual logic — the loaders (`load_news`, `load_twitter`,
`load_irony`, `load_figlang`), the per-dataset cleaner (`clean_one`), the
`DATASETS` registry, and the shared `CleaningLog` / `text_cleaning` helpers —
**stays inside `02_clean_supplementary.py` and the `common/` package**. This
notebook does not redefine any of it; it only imports the module and calls
its pieces one stage at a time so each step's output can be inspected in
between.

Unified schema produced at the end:

| column    | type  | meaning                                             |
|-----------|-------|------------------------------------------------------|
| `dataset` | str   | `news_headlines` \| `twitter_sarcasm` \| `tweeteval_irony` \| `figlang_*` |
| `split`   | str   | `train` \| `validation` \| `test`                   |
| `platform`| str   | `news` \| `twitter` \| `reddit`                     |
| `label`   | int8  | 1 = sarcastic/ironic, 0 = not                        |
| `text`    | str   | cleaned target text                                  |
| `context` | str   | cleaned conversational context (`""` when none)      |
| ...       |       | + the same surface features as the primary corpus    |

Pipeline stages:
1. load each raw corpus into the shared `(split, label, text, context)` shape
2. clean each one independently with `clean_one` (leak stripping, length /
   language filters, de-dup, cross-split leakage removal)
3. save each cleaned corpus (parquet + CSV sample + JSON audit report)
4. concatenate all cleaned corpora into one combined supplementary table
5. print a quick label-balance overview


## Setup

Import the pipeline module and its dependencies. Nothing here is redefined — everything comes straight from `02_clean_supplementary.py` and `common/`.

In [1]:
from __future__ import annotations

import importlib.util
import sys
from pathlib import Path

import pandas as pd

# `scripts/` holds both `common/` and the numbered pipeline scripts.
# Adjust this if the notebook lives somewhere other than pipeline/notebooks/.
SCRIPTS_DIR = Path.cwd() / ".."
sys.path.insert(0, str(SCRIPTS_DIR.resolve()))

from common.paths import (
    CLEAN_REPORTS, PROCESSED, RAW_FIGLANG, RAW_IRONY, RAW_NEWS, RAW_TWEETS,
    SAMPLES, ensure_dirs,
)
from common.report import CleaningLog
from common import text_cleaning as tc

# The script's filename starts with a digit ("02_..."), so it can't be
# imported with a normal `import` statement — load it by file path instead.
_spec = importlib.util.spec_from_file_location(
    "clean_supplementary", SCRIPTS_DIR / "02_clean_supplementary.py"
)
csup = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(csup)

ensure_dirs()
pd.set_option("display.max_colwidth", 120)

USE_LANGDETECT = True   # mirrors the --no-langdetect CLI flag (True = langdetect enabled)
ONLY = None              # e.g. ["news_headlines", "twitter_sarcasm"]; None = all datasets

wanted = ONLY or list(csup.DATASETS)
print("Datasets to process:", wanted)


Datasets to process: ['news_headlines', 'twitter_sarcasm', 'tweeteval_irony', 'figlang_reddit', 'figlang_twitter']


## Step 1 — load each raw corpus

Each loader returns a `DataFrame` with the shared `(split, label, text, context)` columns, using the raw files under `RAW_NEWS` / `RAW_TWEETS` / `RAW_IRONY` / `RAW_FIGLANG`.

In [2]:
raw_frames: dict[str, pd.DataFrame] = {}

for name in wanted:
    platform, loader = csup.DATASETS[name]
    print(f"[load] {name} ({platform}) ...")
    raw_frames[name] = loader()
    print(f"       {len(raw_frames[name]):,} rows")

raw_frames[wanted[0]].head()


[load] news_headlines (news) ...
       55,328 rows
[load] twitter_sarcasm (twitter) ...
       5,000 rows
[load] tweeteval_irony (twitter) ...
       4,601 rows
[load] figlang_reddit (reddit) ...
       6,200 rows
[load] figlang_twitter (twitter) ...
       6,800 rows


,split,label,text,context
0,train,1,thirtysomething scientists unveil doomsday clock of hair loss,
1,train,0,"dem rep. totally nails why congress is falling short on gender, racial equality",
2,train,0,eat your veggies: 9 deliciously different recipes,
3,train,1,inclement weather prevents liar from getting to work,
4,train,1,mother comes pretty close to using word 'streaming' correctly,


## Step 2 — clean each corpus

`clean_one` runs the full per-dataset pipeline: drop unlabelled/empty rows, normalise + de-leak text via `text_cleaning`, enforce a minimum word count, filter to English, de-duplicate within and across splits, and tag the result with `dataset` / `platform`. It returns both the cleaned `DataFrame` and a `CleaningLog` you can inspect.

In [3]:
cleaned: dict[str, pd.DataFrame] = {}
logs: dict[str, CleaningLog] = {}

for name in wanted:
    platform, _ = csup.DATASETS[name]
    print(f"\n=== {name} ({platform}) ===")
    df, log = csup.clean_one(
        name, platform, raw_frames[name], use_langdetect=USE_LANGDETECT
    )
    cleaned[name] = df
    logs[name] = log
    log.print_table()

cleaned[wanted[0]].head()



=== news_headlines (news) ===
  [news_headlines] normalising text ...

--- cleaning audit: news_headlines ---
step reason                                   removed    %orig        left
1    no usable label                                0    0.00%      55,328
2    empty / placeholder text                       0    0.00%      55,328
3    empty after markup/leak removal                0    0.00%      55,328
4    fewer than 2 words                             2    0.00%      55,326
5    non-English                                  249    0.45%      55,077
6    exact dup within split                       225    0.41%      54,852
7    same text carries both labels                  0    0.00%      54,852
8    duplicate across splits (test copy dropped)    26,477   47.85%      28,375
     TOTAL RETAINED                                     51.29%      28,375

=== twitter_sarcasm (twitter) ===
  [twitter_sarcasm] normalising text ...

--- cleaning audit: twitter_sarcasm ---
step reason      

,dataset,platform,split,label,text,context,text_clean,context_clean,n_chars,n_words,...,n_urls,n_mentions,n_hashtags,n_interjections,starts_with_interjection,context_n_words,has_context,en_score,is_english,lang_method
0,news_headlines,news,train,1,thirtysomething scientists unveil doomsday clock of hair loss,,thirtysomething scientists unveil doomsday clock of hair loss,,61,8,...,0,0,0,0,False,0,False,0.759375,True,heuristic-high
1,news_headlines,news,train,0,"dem rep. totally nails why congress is falling short on gender, racial equality",,"dem rep. totally nails why congress is falling short on gender, racial equality",,79,13,...,0,0,0,1,False,0,False,0.784615,True,heuristic-high
2,news_headlines,news,train,0,eat your veggies: 9 deliciously different recipes,,eat your veggies: 9 deliciously different recipes,,49,6,...,0,0,0,0,False,0,False,0.795833,True,heuristic-high
3,news_headlines,news,train,1,inclement weather prevents liar from getting to work,,inclement weather prevents liar from getting to work,,52,8,...,0,0,0,0,False,0,False,0.978125,True,heuristic-high
4,news_headlines,news,train,1,mother comes pretty close to using word 'streaming' correctly,,mother comes pretty close to using word 'streaming' correctly,,61,9,...,0,0,0,0,False,0,False,0.747222,True,heuristic-high


## Step 3 — save per-dataset outputs

For each dataset: the cleaned parquet, a 300-row CSV sample for manual inspection, and the JSON cleaning-audit report.

In [4]:
for name in wanted:
    df = cleaned[name]
    log = logs[name]

    out = PROCESSED / f"{name}_clean.parquet"
    df.to_parquet(out, index=False)
    print(f"[save] {out}  ({len(df):,} rows)")

    sample_path = SAMPLES / f"{name}_head300.csv"
    df.head(300).to_csv(sample_path, index=False, encoding="utf-8")
    print(f"[save] {sample_path}")

    report_path = CLEAN_REPORTS / f"{name}_cleaning_report.json"
    log.save(report_path)
    print(f"[save] {report_path}")


[save] C:\Users\USER\OneDrive\Documents\CS3244\cs3244_group1\data\processed\news_headlines_clean.parquet  (28,375 rows)
[save] C:\Users\USER\OneDrive\Documents\CS3244\cs3244_group1\data\samples\news_headlines_head300.csv
[save] C:\Users\USER\OneDrive\Documents\CS3244\cs3244_group1\results\cleaning\news_headlines_cleaning_report.json
[save] C:\Users\USER\OneDrive\Documents\CS3244\cs3244_group1\data\processed\twitter_sarcasm_clean.parquet  (224 rows)
[save] C:\Users\USER\OneDrive\Documents\CS3244\cs3244_group1\data\samples\twitter_sarcasm_head300.csv
[save] C:\Users\USER\OneDrive\Documents\CS3244\cs3244_group1\results\cleaning\twitter_sarcasm_cleaning_report.json
[save] C:\Users\USER\OneDrive\Documents\CS3244\cs3244_group1\data\processed\tweeteval_irony_clean.parquet  (4,583 rows)
[save] C:\Users\USER\OneDrive\Documents\CS3244\cs3244_group1\data\samples\tweeteval_irony_head300.csv
[save] C:\Users\USER\OneDrive\Documents\CS3244\cs3244_group1\results\cleaning\tweeteval_irony_cleaning_repor

## Step 4 — combine into one supplementary table

Concatenate the cleaned datasets, keeping only the shared columns (`FEATURE_COLS` plus the schema columns), and write the combined parquet + summary JSON.

In [5]:
keep = (["dataset", "platform", "split", "label", "text_clean",
         "context_clean", "has_context", "context_n_words",
         "n_context_turns"] + csup.FEATURE_COLS)

combined = [cleaned[name][[c for c in keep if c in cleaned[name].columns]]
            for name in wanted]
allsup = pd.concat(combined, ignore_index=True)

out = PROCESSED / "supplementary_all_clean.parquet"
allsup.to_parquet(out, index=False)
print(f"[save] combined -> {out}  ({len(allsup):,} rows)")

import json
summary = [logs[name].to_dict() for name in wanted]
summary_path = CLEAN_REPORTS / "supplementary_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)
print(f"[save] {summary_path}")


[save] combined -> C:\Users\USER\OneDrive\Documents\CS3244\cs3244_group1\data\processed\supplementary_all_clean.parquet  (45,780 rows)
[save] C:\Users\USER\OneDrive\Documents\CS3244\cs3244_group1\results\cleaning\supplementary_summary.json


## Step 5 — quick overview

Label balance per dataset / platform, as a sanity check before moving on to feature engineering / modelling.

In [6]:
allsup.groupby(["dataset", "platform", "label"]).size().unstack(fill_value=0)


,label,0,1
dataset,platform,,
figlang_reddit,reddit,3077,3067
figlang_twitter,twitter,3146,3308
news_headlines,news,14894,13481
tweeteval_irony,twitter,2375,2208
twitter_sarcasm,twitter,112,112
